Jupyter notebook to train a ramdom classifier ML model for anamaly detection of Network traffic

## **1. Download packages required to download a kaggle dataset into the notebook**

In [ ]:
!pip install opendatasets
!pip install pandas

In [ ]:
!pip install kaggle

In [ ]:
mkdir -p ~/.kaggle && echo KGAT_3511548bd9a061411359385a429f4050 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

### **Download kaggle Network Traffic Anomaly Detection Dataset**

In [ ]:
!kaggle datasets download rebsonramalho/network-threat-detection-dataset

Dataset URL: https://www.kaggle.com/datasets/rebsonramalho/network-threat-detection-dataset
License(s): unknown
network-threat-detection-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


### **Unzip the dataset zip file**

In [ ]:
!unzip network-threat-detection-dataset.zip

Archive:  network-threat-detection-dataset.zip
replace Dataset_completo.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import pandas as pd

network_data = pd.read_csv('Dataset_completo.csv', low_memory=False)

In [ ]:
display(network_data)

## **Data preprocessing**

### **Drop all rows with any null columns**

In [ ]:
network_data.dropna()

In [ ]:
# 1. Remove duplicates
network_data_clean = network_data.drop_duplicates()
display(network_data_clean)

In [ ]:
# 2. drop rows where trans_depth/res_bdy_len contain HTTP junk (GET, /paths)
bad_rows = (
    pd.to_numeric(network_data_clean["trans_depth"], errors="coerce").isna() & network_data_clean["trans_depth"].notna()
) | (
    pd.to_numeric(network_data_clean["res_bdy_len"], errors="coerce").isna() & network_data_clean["res_bdy_len"].notna()
)
print(f"Dropping {bad_rows.sum():,} corrupted rows")
network_data_clean = network_data_clean[~bad_rows]

In [ ]:
display(network_data_clean)

## **Remove all unneeded columns with categoral data:**
### Machine learning models can only work with numerical data and requires additional encoding and preprocessing of categorical in order to be used by the model

In [ ]:
network_data_clean.dtypes

# Columns We have decided to drop and why:

**is_sm_ips_ports** - Constant 0 on every row — zero information

**trans_depth** - No valid numeric values; ~21k rows contain HTTP methods like GET in the wrong column

**res_bdy_len** - Same problem — contains URL paths like /dvwa/login.php, not numeric data

**Ltime** - Fully redundant with Stime + dur (100% match in your data)

**Sload** - Derived from sbytes / dur (correlation ≈ 1.0)

**Dload** Derived from dbytes / dur

**srcip, dstip** - Drop raw IPs for generalizable threat detection. Your dataset only has ~21 source and ~44 destination IPs in a fixed lab (192.168.56.x), so the model will memorize topology instead of attack behavior. Better: derive features like same_subnet,

**is_private** - is_multicast if you need IP context.
**Stime** - Drop raw timestamp unless you engineer features (hour, day_of_week). Raw timestamps overfit to one capture window.

**synack** - Weak feature — only True when present, ~45% missing. Safe to drop unless you’re doing TCP-specific analysis.

**ackdat** - ~74% missing. Low value; drop unless TCP handshake timing matters.

In [ ]:
DROP_COLS = [
    "is_sm_ips_ports", "trans_depth", "res_bdy_len",
    "Ltime", "Sload", "Dload",
    "synack", "ackdat",
    "srcip", "dstip", "Stime",    # drop raw identifiers/timestamps
]

In [ ]:
network_data_clean.drop(columns=DROP_COLS, inplace=True)

In [ ]:
display(network_data_clean)

In [ ]:
network_data_clean.dtypes

# Chosen Features and why:

**sport** - Client-side port choice
Context for connection origin

**dsport** - Target service/port
Attack type & target selection

**proto** - L3/L4 protocol
Separates TCP/UDP/ICMP behavior

**state** - Connection phase
Scans vs completed sessions

**dur** - Session length
Timing of attack vs normal use

**sbytes / dbytes** - Data volume each direction
Exfil, probes, vs real responses

**Spkts / Dpkts** - Packet counts each direction
Scan bursts, conversation balance

**smeansz / dmeansz** - Average packet sizes
Payload shape without content

**service** - Inferred application
Strong shortcut for http vs attack traffic

**ct_state_ttl** - Detailed TCP state
SYN scans, half-open connections

# Encoding Categorical data
The following columns are essential to the model's classification of network traffic and will need to be encoded into numerical values for use by the model.

1. proto
2. state
3. service
4. ct_state_ttl

# Format columns so that all the data is uniform so as to not confuse the model

**proto**

Different export sources recorded the same protocol as a name (TCP, UDP) or an IANA number (6, 17.0). Without fixing this, one-hot encoding treats 17.0 and UDP as different categories, splitting the same protocol across two features and weakening the model.

**state**

About some rows use 0 instead of a real state like new or closed. That 0 isn’t a valid connection state — it means unknown/missing. If you leave it, the model learns a fake category "0" that doesn’t mean anything in network analysis.

**service**

Same issue as state: 0 means “no service detected”, not a real service name. Rows with real values use names like http, dns, failed. Mapping 0 → "unknown" keeps missing data honest and avoids a misleading one-hot column.

**ct_state_ttl**

Again, 0 is a placeholder for missing TCP state info, while real rows use labels like syn_sent, closed, fin_wait2. Normalizing prevents the encoder from treating “missing” as a real TCP state.

**sport and dsport**

Ports are stored as floats (80.0, 53.0) because pandas read them from CSV that way. A port is logically an integer — converting avoids odd float behavior and makes values easier to interpret (port 80, not 80.0).

**sttl**

Almost all values are numeric TTL hop counts, but a few rows contain invalid strings like 64,64 (likely a merge/parsing error). Coercing to numeric turns those into NaN instead of breaking the model or dtype conversion later.

**Label**

Should be a clean integer target (0 or 1). Casting with .astype(int) avoids accidental float labels like 1.0 and ensures scikit-learn treats it correctly as a classification target.

Used generative AI to help with formatting

In [ ]:

CAT_COLS = ["proto", "state", "service", "ct_state_ttl"]
NUM_COLS = [
    "sport", "dsport", "dur", "sbytes", "dbytes", "sttl",
    "Spkts", "Dpkts", "swin", "stcpb", "dtcpb", "smeansz", "dmeansz",
]

PROTO_MAP = {
    "6": "TCP", "6.0": "TCP", "TCP": "TCP",
    "17": "UDP", "17.0": "UDP", "UDP": "UDP",
    "1": "ICMP", "1.0": "ICMP", "ICMP": "ICMP",
    "58": "IPv6-ICMP", "58.0": "IPv6-ICMP", "IPv6-ICMP": "IPv6-ICMP",
    "2": "IGMP", "2.0": "IGMP",
    "0": "unknown", "0.0": "unknown",
}

def normalize_proto(val):
    s = str(val).strip()
    if s.endswith(".0"):
        s = s[:-2]
    if "," in s:
        return "unknown"
    return PROTO_MAP.get(s, "unknown")
# --- proto: numbers + names -> canonical names ---
network_data_clean["proto"] = network_data_clean["proto"].map(normalize_proto)
# --- state, service, ct_state_ttl: 0 -> unknown ---
for col in ["state", "service", "ct_state_ttl"]:
    network_data_clean[col] = (
        network_data_clean[col]
        .astype(str)
        .replace({"0": "unknown", "0.0": "unknown", "nan": "unknown"})
    )
# --- numeric columns ---
for col in NUM_COLS:
    network_data_clean[col] = pd.to_numeric(network_data_clean[col], errors="coerce")

After normalising columns so that is doesn't have contradictory values


1. Split the data into training, validation and testing sets
2. Encode the training data


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [ ]:

X, Y = network_data_clean.loc[ : , (network_data_clean.columns != "Label")], network_data_clean["Label"]
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.25, random_state=42
)

# --- preprocessing (encoder) ---
encoder = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ]
)

X_train = encoder.fit_transform(X_train)
X_val = encoder.transform(X_val)
X_test = encoder.transform(X_test)
feature_names = encoder.get_feature_names_out()
X_train = pd.DataFrame(X_train, columns=feature_names, index=Y_train.index)
X_val = pd.DataFrame(X_val, columns=feature_names, index=Y_val.index)
X_test = pd.DataFrame(X_test, columns=feature_names, index=Y_test.index)

# Training the random forest classifier model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
default_rf_model = RandomForestClassifier(
    class_weight="balanced",   # helpful because Label is imbalanced
    random_state=42,
    n_jobs=-1,
)
default_rf_model.fit(X_train, Y_train)

# Validating the default_rf_model using validation data

In [ ]:
Y_pred = default_rf_model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(Y_val, Y_pred):.4f}")
print("\nClassification Report\n", classification_report(Y_val, Y_pred))
print("\nConfusion Matrix\n", confusion_matrix(Y_val, Y_pred))

# Training another ML model XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

In [ ]:
scale_pos_weight = (Y_train == 0).sum() / (Y_train == 1).sum()

# The data (X_train) is already encoded
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

# train on already preprocessed features
xgb_model.fit(X_train, Y_train)

# Validate XGBOOST ML model

In [ ]:
y_pred = xgb_model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(Y_val, y_pred):.4f}")
print("\nClassification Report\n", classification_report(Y_val, y_pred))
print("\nConfusion Matrix\n", confusion_matrix(Y_val, y_pred))

use shap to create an explainer model

In [ ]:
import shap
import numpy as np

In [ ]:
# sample validation data for speed
sample_size = min(2000, len(X_val))
X_sample = X_val.sample(n=sample_size, random_state=42)

# TreeExplainer for Random Forest
explainer = shap.TreeExplainer(default_rf_model)
shap_values = explainer.shap_values(X_sample)

# binary classification: use class 1 (malicious)
# shap_values is a 3D array (n_samples, n_features, n_classes)
# Select SHAP values for class 1 (index 1)
shap_values_class_1 = shap_values[:, :, 1]

# top 5 contributors
top5 = (
    pd.Series(np.abs(shap_values_class_1).mean(axis=0), index=X_sample.columns)
    .sort_values(ascending=False)
    .head(5)
)
print("Top 5 contributors:\n")
print(top5)
# bar plot
shap.summary_plot(
    shap_values_class_1,
    X_sample,
    plot_type="bar",
    max_display=5,
    show=True,
)

In [ ]:
import pickle
from google.colab import files


# export rf model and tree explainer
rf_filename = "rf_model.pkl"
tree_explainer_filename = "tree_explainer.pkl"

with open(rf_filename, "wb") as file:
    pickle.dump(default_rf_model, file)


with open(tree_explainer_filename, "wb") as file:
    pickle.dump(explainer, file)
# joblib.dump(default_rf_model, rf_filename)
# joblib.dump(explainer, tree_explainer_filename)


# download models
files.download(rf_filename)
files.download(tree_explainer_filename)

# export cleaned test data
X_test.to_csv('clean_test_data.csv', index=False)
files.download('clean_test_data.csv')